In [38]:
!apt-get -qq update
!apt-get -qq install -y poppler-utils
!pip -q install pdf2image pillow

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


# Convert PDF to Image

In [ ]:
from pdf2image import convert_from_path
from PIL import Image

pages = convert_from_path("/content/drilling report.pdf", dpi=300)

# Ensure consistent mode (RGB) and width
pages = [p.convert("RGB") for p in pages]
max_w = max(p.width for p in pages)

# Optionally pad pages to same width (so alignment is nice)
padded = []
for p in pages:
    if p.width != max_w:
        canvas = Image.new("RGB", (max_w, p.height), (255, 255, 255))
        canvas.paste(p, ((max_w - p.width)//2, 0))
        padded.append(canvas)
    else:
        padded.append(p)

total_h = sum(p.height for p in padded)
long_img = Image.new("RGB", (max_w, total_h), (255, 255, 255))

y = 0
for p in padded:
    long_img.paste(p, (0, y))
    y += p.height

out_path = "drilling report.png"
long_img.save(out_path)
out_path

'drilling report.png'

# Vision Task

In [1]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import csv

img_path = "drilling report.png"
img = cv2.imread(img_path)

if img is None:
    raise FileNotFoundError(f"Couldn't read {img_path}. Check the filename in /content.")

print("Image shape (H, W, C):", img.shape)

Image shape (H, W, C): (68461, 3091, 3)


In [2]:
def find_next_line_boundaries(bw_normalized, prev_line_end_boundary, is_horizontal=True):
    # Transpose if searching for vertical lines
    processing_array = bw_normalized
    if not is_horizontal:
        processing_array = np.transpose(bw_normalized)

    total_dimension = processing_array.shape[0] # Number of rows

    # Search for the next line
    in_line = False
    start_boundary = None
    end_boundary = None

    # Start searching *after* the previous line's end boundary
    search_start_index = prev_line_end_boundary + 1

    # Ensure search_start_index is within bounds
    if search_start_index >= total_dimension:
        print(f"Error: Search must start before total dimension ({total_dimension}). Previous line end was {prev_line_end_boundary}.")
        return None, None

    for current_boundary in range(search_start_index, total_dimension):
        is_solid_black = np.all(processing_array[current_boundary]) # current_boundary is now a column index

        if is_solid_black and not in_line:
            # Found the start of a new line
            in_line = True
            start_boundary = current_boundary

        elif not is_solid_black and in_line:
            # Found the end of the line
            end_boundary = current_boundary - 1
            # We've found the *first* line after prev_line_end_boundary, so we're done.
            return start_boundary, end_boundary

        # If loop finishes and we are still in_line, it means the line extends to the end
        if current_boundary == total_dimension - 1 and in_line:
            end_boundary = total_dimension - 1
            return start_boundary, end_boundary

    # If the loop finishes without finding an end, it means no new line was found after prev_line_end_boundary
    print(f"No subsequent solid black line found after boundary {prev_line_end_boundary}.")
    return None, None


## Extract Lithology & Drilling Rate Columns

In [3]:
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
_, mask_bw = cv2.threshold(gray, 220, 255, cv2.THRESH_BINARY_INV)
bw_normalized = (mask_bw > 0).astype(np.uint8)

_, first_line_last_row = find_next_line_boundaries(bw_normalized, -1)
_, second_line_last_row = find_next_line_boundaries(bw_normalized, first_line_last_row)
third_line_start_row, third_line_last_row = find_next_line_boundaries(bw_normalized, second_line_last_row)
_, fourth_line_last_row = find_next_line_boundaries(bw_normalized, third_line_last_row)

print(f"Third line starts at row: {third_line_start_row}")
print(f"Fourth line ends at row: {fourth_line_last_row}")

# Extract the 'headers' image segment
headers = None
if third_line_start_row is not None and fourth_line_last_row is not None:
    headers = img[third_line_start_row : fourth_line_last_row + 1, :]

    print("Image segment 'headers' extracted successfully.")
else:
    print("Could not find both the 3rd and 4th horizontal lines with the specified criteria.")
    

Third line starts at row: 2131
Fourth line ends at row: 2399
Image segment 'headers' extracted successfully.


In [4]:
gray = cv2.cvtColor(headers, cv2.COLOR_BGR2GRAY)
_, mask_bw = cv2.threshold(gray, 220, 255, cv2.THRESH_BINARY_INV)
bw_normalized = (mask_bw > 0).astype(np.uint8)
first_v_line_start_col, first_v_line_end_col = find_next_line_boundaries(bw_normalized, -1, is_horizontal=False)
second_v_line_start_col, second_v_line_end_col = find_next_line_boundaries(bw_normalized, first_v_line_end_col, is_horizontal=False)
third_v_line_start_col, third_v_line_end_col = find_next_line_boundaries(bw_normalized, second_v_line_end_col, is_horizontal=False)
fourth_v_line_start_col, fourth_v_line_end_col = find_next_line_boundaries(bw_normalized, third_v_line_end_col, is_horizontal=False)

print(f"First vertical line starts at column: {first_v_line_start_col}")
print(f"Second vertical line ends at column: {second_v_line_end_col}")
print(f"Third vertical line starts at column: {third_v_line_start_col}")
print(f"Fourth vertical line ends at column: {fourth_v_line_end_col}")


First vertical line starts at column: 0
Second vertical line ends at column: 431
Third vertical line starts at column: 494
Fourth vertical line ends at column: 804


In [5]:
lithology_img = img[fourth_line_last_row+1:, third_v_line_start_col:fourth_v_line_end_col]
drillRate_img = img[fourth_line_last_row+1:, first_v_line_end_col+1:second_v_line_start_col]

## Process Lithology

In [6]:
def max_line_transitions(binary_img, direction="horizontal"):
    img = (binary_img > 0).astype(np.int16)

    if direction == "horizontal":
        transitions = np.abs(np.diff(img, axis=1))
        per_line = np.sum(transitions, axis=1)

    elif direction == "vertical":
        transitions = np.abs(np.diff(img, axis=0))
        per_line = np.sum(transitions, axis=0)

    else:
        raise ValueError("direction must be 'horizontal' or 'vertical'")

    return int(np.max(per_line))

def dominant_hsv(patch_bgr: np.ndarray, mask: np.ndarray = None):
    hsv = cv2.cvtColor(patch_bgr, cv2.COLOR_BGR2HSV)
    if mask is None:
        pixels = hsv.reshape(-1, 3)
    else:
        pixels = hsv[mask > 0].reshape(-1, 3)

    if pixels.shape[0] == 0:
        return (0, 0, 0)
    
    start_range = 170
    end_range = 179

    # Create a mask for values within the specified range
    mask = (pixels[:, 0] >= start_range) & (pixels[:, 0] <= end_range)

    # Apply the transformation only to the elements within the mask
    # The transformation is: 9 - (value - 170)
    pixels[mask, 0] = 9 - (pixels[mask, 0] - start_range)

    h = int(np.median(pixels[:, 0]))
    s = int(np.median(pixels[:, 1]))
    v = int(np.median(pixels[:, 2]))
    return (h, s, v)

In [7]:
V_BLACK_MAX = 60          # background considered black if V <= this
S_YELLOW_MIN = 70         # yellow typically has decent saturation
V_YELLOW_MIN = 120        # yellow is bright-ish
H_YELLOW_RANGE = (15, 40) # yellow hue range in OpenCV HSV (0..179)

# Foreground line color thresholds
# We'll compute dominant HSV on "non-background" pixels.
H_BLUE_RANGE  = (85, 135)
H_GREEN_RANGE = (35, 85)
# Red wraps around 0: (0..10) U (170..179)
H_RED_LOW     = (0, 10)
H_RED_HIGH    = (170, 179)

V_DARK_MAX = 70           # for black-ish foreground lines


def classify_background(cell_bgr: np.ndarray) -> str:
    h, s, v = dominant_hsv(cell_bgr)

    if v <= V_BLACK_MAX:
        return "black"

    # yellow: hue in yellow band, moderate/high sat, bright
    if (H_YELLOW_RANGE[0] <= h <= H_YELLOW_RANGE[1]) and (s >= S_YELLOW_MIN) and (v >= V_YELLOW_MIN):
        return "yellow"

    return "white"


def foreground_mask(cell_bgr: np.ndarray, bg: str) -> np.ndarray:
    hsv = cv2.cvtColor(cell_bgr, cv2.COLOR_BGR2HSV)

    if bg == "black":
        # foreground likely brighter than black background
        # take pixels with V above a threshold
        mask = (hsv[:, :, 2] > (V_BLACK_MAX + 20)).astype(np.uint8) * 255
        return mask

    if bg == "yellow":
        # remove yellow-ish pixels to keep lines
        h, s, v = hsv[:, :, 0], hsv[:, :, 1], hsv[:, :, 2]
        yellow_bg = ((h >= H_YELLOW_RANGE[0]) & (h <= H_YELLOW_RANGE[1]) & (s >= 40) & (v >= 80))
        mask = (~yellow_bg).astype(np.uint8) * 255
        return mask

    # bg == "white": remove near-white pixels
    # white tends to have low S and high V
    h, s, v = hsv[:, :, 0], hsv[:, :, 1], hsv[:, :, 2]
    white_bg = (s < 150) & (v > 200)
    mask = (~white_bg).astype(np.uint8) * 255
    return mask


def classify_foreground_color(cell_bgr: np.ndarray, bg: str) -> str:
    mask = foreground_mask(cell_bgr, bg)
    if np.sum(mask) == 0:
      return "no_foreground"

    h, s, v = dominant_hsv(cell_bgr, mask=mask)

    # black lines: low V
    if v <= V_DARK_MAX or (v <= 2*V_DARK_MAX and s <= 50):
        return "black"

    # red wraps around
    if (H_RED_LOW[0] <= h <= H_RED_LOW[1]) or (H_RED_HIGH[0] <= h <= H_RED_HIGH[1]):
        return "red"

    if H_BLUE_RANGE[0] <= h <= H_BLUE_RANGE[1]:
        return "blue"

    if H_GREEN_RANGE[0] <= h <= H_GREEN_RANGE[1]:
        return "green"

    # fallback if ambiguous
    return "unknown"

In [8]:
def binarize_foreground01(cell_bgr: np.ndarray, bg: str) -> np.ndarray:
    gray = cv2.cvtColor(cell_bgr, cv2.COLOR_BGR2GRAY)

    if bg == "white":
        # foreground usually darker than background -> invert so foreground becomes 1
        _, bw = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    elif bg == "yellow":
        th, _ = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
        _, bw = cv2.threshold(gray, 0.5*th, 255, cv2.THRESH_BINARY_INV)
    else:
        _, bw = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    k = cv2.getStructuringElement(cv2.MORPH_RECT, (3,3))
    bw = cv2.morphologyEx(bw, cv2.MORPH_CLOSE, k,iterations=1)
    return (bw > 0).astype(np.uint8)


def binary_area(binary01: np.ndarray) -> int:
    return int(binary01.sum())

In [9]:
def classify_symbol(cell_bgr: np.ndarray) -> str:
    bg = classify_background(cell_bgr)

    # 1) background black -> OBSIDIAN
    if bg == "black":
        return "OBSIDIAN"

    b01 = binarize_foreground01(cell_bgr, bg)

    # 2) background yellow
    if bg == "yellow":
        t = max_line_transitions(b01, direction="vertical")
        if t == 2:
            a = binary_area(b01)
            if a < 80:
                return "SAND"
            elif a < 150:
                return "MEDIUM SAND"
            else:
                return "COARSE SAND"
        elif t == 4:
            return "CONGLOMERATE"

    # 3) background white
    fg_color = classify_foreground_color(cell_bgr, bg="white")

    if fg_color == "no_foreground":
        return "blank"

    if fg_color == "black":
        t = max_line_transitions(b01[:b01.shape[0]//5, :], direction="horizontal")
        if t == 0:
            return "DIORITE"
        else:
            return "CEMENT"

    if fg_color == "blue":
        t = max_line_transitions(b01, direction="vertical")
        if t == 2:
            return "MONZONITE"
        else:
            return "DACITE"

    if fg_color == "red":
        t = max_line_transitions(b01, direction="horizontal")
        if t == 2:
            return "GRANODIORITE"
        else:
            return "GRANITE"

    if fg_color == "green":
        t = max_line_transitions(b01, direction="horizontal")
        if t < 4:
            t2 = max_line_transitions(b01, direction="vertical")
            if t2 == 2:
                return "CLAY"
            if t2 == 4:
                return "CLAYSTONE"
        elif t == 4:
            return "RHYOLITE"
        else:
            return "GRANITE WASH"
        
    return "UNKNOWN"

In [10]:
CSV_COLUMNS = [
    "Depth_From (m)", "Depth_To (m)",
    "OBSIDIAN","RHYOLITE","DACITE","MONZONITE","GRANODIORITE","DIORITE",
    "CLAYSTONE","CLAY","CEMENT","SAND","CONGLOMERATE","GRANITE WASH",
    "GRANITE","MEDIUM SAND","COARSE SAND", "UNKNOWN"
]

SYMBOL_CLASSES = CSV_COLUMNS[2:]  # the lithology columns

def extract_column_boundaries(img):
    column_boundaries = []
    _, start_col = find_next_line_boundaries(img, -1, is_horizontal=False)

    # Loop to find all vertical lines (columns)
    while True:
        end_col, start_next_col = find_next_line_boundaries(img, start_col, is_horizontal=False)

        if start_col is not None and end_col is not None:
            # Found a vertical line (column)
            column_boundaries.append((start_col, end_col))
            start_col = start_next_col
        else:
            return column_boundaries
        
def extract_table_mask(img):
    img_gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    _, bw = cv2.threshold(img_gray, 100, 255, cv2.THRESH_BINARY_INV)

    bw_normalized = (bw > 0).astype(np.uint8)

    k = cv2.getStructuringElement(cv2.MORPH_RECT, (3,3))
    mask_h = cv2.morphologyEx(bw_normalized, cv2.MORPH_DILATE, k, iterations=1)
    k = cv2.getStructuringElement(cv2.MORPH_RECT, (3,1))
    mask_h = cv2.morphologyEx(mask_h, cv2.MORPH_ERODE, k, iterations=10)
    k = cv2.getStructuringElement(cv2.MORPH_RECT, (1,3))
    mask_h = cv2.morphologyEx(mask_h, cv2.MORPH_ERODE, k, iterations=1)
    k = cv2.getStructuringElement(cv2.MORPH_RECT, (3,3))
    mask_h = cv2.morphologyEx(mask_h, cv2.MORPH_DILATE, k, iterations=1)

    k = cv2.getStructuringElement(cv2.MORPH_RECT, (3,3))
    mask_v = cv2.morphologyEx(bw_normalized, cv2.MORPH_DILATE, k, iterations=1)
    k = cv2.getStructuringElement(cv2.MORPH_RECT, (1,3))
    mask_v = cv2.morphologyEx(mask_v, cv2.MORPH_ERODE, k, iterations=15)

    return cv2.bitwise_or(mask_v, mask_h)

def read_table_to_csv(img_bgr: np.ndarray, out_csv_path: str,
                      depth_start: int = 50,
                      depth_step: int = 5,
                      max_rows: int = 10000,
                      debug_first_n: int = 0):
    
    table_mask = extract_table_mask(img_bgr)
    column_boundaries = extract_column_boundaries(table_mask[:20, :])
    n_cols = len(column_boundaries)

    print(f"Total number of columns found: {n_cols}")

    start_flag = False
    rows_written = 0
    next_row_line_end = -1

    with open(out_csv_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=CSV_COLUMNS)
        writer.writeheader()

        for j in range(max_rows):
            last_row_line_end = next_row_line_end
            next_row_line_start, next_row_line_end = find_next_line_boundaries(table_mask, last_row_line_end)

            # aggregate counts
            counts = {k: 0 for k in SYMBOL_CLASSES}
            symbols_this_row = []

            for i in range(n_cols):
                cell = img_bgr[last_row_line_end+1:next_row_line_start, column_boundaries[i][0]+1:column_boundaries[i][1]]
                sym = classify_symbol(cell)
                symbols_this_row.append(sym)
                if sym in counts:
                    counts[sym] += 1

            if not start_flag:
                if any(s == "blank" for s in symbols_this_row):
                    continue
                else:
                    start_flag = True

            elif all(s == "blank" for s in symbols_this_row):
              break

            depth_from = depth_start + j * depth_step
            depth_to = depth_from + depth_step

            row = {"Depth_From (m)": depth_from, "Depth_To (m)": depth_to}
            for k in SYMBOL_CLASSES:
                row[k] = counts[k] * (100 // n_cols)

            writer.writerow(row)
            rows_written += 1

            if debug_first_n and j < debug_first_n:
                print(f"Row {j} depth {depth_from}-{depth_to}: {symbols_this_row} -> {row}")

    return rows_written

In [11]:
out_csv = "Lithology.csv"
N_ROWS = read_table_to_csv(lithology_img, out_csv_path=out_csv, debug_first_n=3)
print("Rows written:", N_ROWS)
print("Saved:", out_csv)

Error: Search must start before total dimension (310). Previous line end was 309.
Total number of columns found: 10
Rows written: 2172
Saved: Lithology.csv


## Process Drill Rate


In [12]:
h, w = drillRate_img.shape[:2]
b, g, r = cv2.split(drillRate_img)

# Blue dominant mask
mask_blue = (b > 120) & (r < 50) & (g < 50)

# Extract x(y): for each row, take median x of detected blue pixels
x_by_y = np.full(h, np.nan, dtype=np.float32)

for y in range(h):
    xs = np.where(mask_blue[y] > 0)[0]
    if xs.size > 0:
        x_by_y[y] = np.median(xs)  # robust center estimate

valid = np.where(~np.isnan(x_by_y))[0]
ymin, ymax = valid.min(), valid.max()
y_range = np.arange(ymin, ymax + 1)

# Fill occluded/missing rows by interpolation
x_interp = np.interp(y_range, valid, x_by_y[valid])


In [13]:
drillRate_img_gray = cv2.cvtColor(drillRate_img[:drillRate_img.shape[0]//500, :], cv2.COLOR_BGR2GRAY)
_, drillRate_bw = cv2.threshold(drillRate_img_gray, 200, 255, cv2.THRESH_BINARY_INV)

drillRate_bw_normalized = (drillRate_bw > 0).astype(np.uint8)

k = cv2.getStructuringElement(cv2.MORPH_RECT, (3, 3))
drillRate_mask_h = cv2.morphologyEx(drillRate_bw_normalized, cv2.MORPH_DILATE, k, iterations=3)
k = cv2.getStructuringElement(cv2.MORPH_RECT, (3,1))
drillRate_mask_h = cv2.morphologyEx(drillRate_mask_h, cv2.MORPH_ERODE, k, iterations=5)
k = cv2.getStructuringElement(cv2.MORPH_RECT, (3,3))
drillRate_mask_h = cv2.morphologyEx(drillRate_mask_h, cv2.MORPH_ERODE, k, iterations=3)

In [14]:
last_line_end = -1
line_starts = []

while True:
    last_line_start, last_line_end = find_next_line_boundaries(drillRate_mask_h, last_line_end)
    if last_line_start is None: break
    line_starts.append(last_line_start)

differences = np.diff(line_starts)

No subsequent solid black line found after boundary 119.


In [15]:
STEP_FT = 5
SCALE = np.mean(differences)
DEPTH_START = int(STEP_FT * (valid[0] - line_starts[0]) / SCALE + 55)
N_ROWS = int(np.ceil(x_interp.shape[0] / SCALE))

out_path = "Drill_Rate.csv"
x_interp = np.asarray(x_interp, dtype=float)

with open(out_path, mode="w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["depth", "weight on bit"])  # CSV header

    for i in range(N_ROWS):
        # Depth range label
        d0 = DEPTH_START + STEP_FT * i
        d1 = d0 + STEP_FT
        depth_label = f"{d0}-{d1}"

        # Slice indices
        start = int(round(i * SCALE))
        end = int(start + SCALE)

        # Safe bounds
        start = max(0, min(start, len(x_interp)))
        end = max(0, min(end, len(x_interp)))

        sl = x_interp[start:end]

        # Mean or NaN if empty
        val = float(np.mean(sl)) if sl.size else np.nan

        # Map: -1 → 0, 0 → 100
        mapped = 100.0 - (100.0 * val / w)
        mapped = float(np.clip(mapped, 0.0, 100.0))

        formatted_mapped = f"{mapped:.2f}"

        writer.writerow([depth_label, formatted_mapped])

print(f"Saved CSV: {out_path}  (rows: {N_ROWS})")

Saved CSV: Drill_Rate.csv  (rows: 2172)
